# Setup env

In [1]:
import pandas as pd
from pathlib import Path

In [23]:
# Get the absolute path of where you are right now
src_path = Path.cwd()
project_src = src_path.parents[1]
print(f"Your project source root is: {project_src}")

# Set 100 rows to display
pd.set_option('display.max_rows', 100) # Set it higher than your 80 rows

Your project source root is: c:\Users\davib\Desktop\MSc_DataScience\DataWarehouse\lab_assignment\imbd


# Read Silver Layer

In [3]:
# Read dims
# bridge_profession_group = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet')
# dim_profession = pd.read_parquet(f'{project_src}\\data\\silver\\dim_profession.parquet')
dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet')
# dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet')
# dim_genres = pd.read_parquet(f'{project_src}\\data\\silver\\dim_genres.parquet')
bridge_genres = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet')
dim_title_basic = pd.read_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet')
bridge_kwn_titles = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet')

# Read fact tables
participations_pers = pd.read_parquet(f'{project_src}\\data\\silver\\participations_pers.parquet')

# Transformations to prepare final data for gold

Analysis to be done:
- Who are the most active people in the film industry? (more participations) OK
- What are the actors with more runtime done? (more runtime considering the participations) OK
- What are the participants with more known participations? (more known titles) NOK
- What are the participants which have worked in more distinct genres? ~~

In [5]:
## Data Structure Exploration

In [10]:
# Explore key tables
print("participations_pers columns:", participations_pers.columns.tolist())
print("dim_person columns:", dim_person.columns.tolist())
print("dim_title_basic columns:", dim_title_basic.columns.tolist())
print("bridge_genres columns:", bridge_genres.columns.tolist())
print("bridge_kwn_titles columns:", bridge_kwn_titles.columns.tolist())
# print("dim_genres columns:", dim_genres.columns.tolist())
# print("dim_roles columns:", dim_roles.columns.tolist())

participations_pers columns: ['nconst', 'primaryName', 'tconst', 'titleType', 'primaryTitle', 'genre_group_id', 'category', 'job', 'characters', 'profession_group_id', 'kwn_title_group_id']
dim_person columns: ['nconst', 'primaryName', 'birthYear', 'deathYear', 'profession_group_id', 'weighting_factor_prf']
dim_title_basic columns: ['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']
bridge_genres columns: ['tconst', 'genre_id', 'genre_group_id', 'weighting_factor_gen']
bridge_kwn_titles columns: ['tconst', 'nconst', 'kwn_title_group_id', 'weighting_factor_grp']


# Filter only movies and tvSeries

## Analysis 1: Most Active People (More Participations)

In [5]:
participations_pers['titleType'].unique()

<ArrowStringArray>
['movie', 'tvSeries']
Length: 2, dtype: str

In [52]:
# Filter for relevant title types
# most_active_people = participations_pers[participations_pers['titleType'].isin(['movie', 'tvSeries'])]

# Count total participations per person and titleType
counts = (
    participations_pers.groupby(['nconst', 'titleType'])
    .size()
    .reset_index(name='total_participations')
)

# Sort by count and then take the top 10 for each titleType
top_10_per_type = (
    counts.sort_values(['titleType', 'total_participations'], ascending=[True, False])
    .groupby('titleType')
    .head(10)
    .merge(dim_person[['nconst', 'primaryName', 'birthYear', 'deathYear']], on='nconst', how='left')
)

top_10_per_type_known =top_10_per_type.merge(bridge_kwn_titles, on='nconst', how='left').merge(dim_title_basic[['tconst', 'primaryTitle', 'isAdult']], on='tconst', how='left')
top_10_per_type_known = top_10_per_type_known[['primaryName', 'total_participations', 'titleType', 'birthYear', 'deathYear', 'primaryTitle', 'isAdult']]
# Display the results
# print("Top 10 Most Active People per Category:")
# print(top_10_per_type[['titleType', 'primaryName', 'total_participations']])
top_10_per_type_known.to_parquet(f'{project_src}\\data\\gold\\active_film_participants.parquet', index=False)

In [24]:
top_10_per_type_known.head(80)

,primaryName,total_participations,titleType,birthYear,deathYear,primaryTitle,isAdult
0,Ilaiyaraaja,1006,movie,1943,<NA>,Sagara Sangamam,0.0
1,Ilaiyaraaja,1006,movie,1943,<NA>,Thalapathi,0.0
2,Ilaiyaraaja,1006,movie,1943,<NA>,Kerala Varma Pazhassi Raja,0.0
3,Ilaiyaraaja,1006,movie,1943,<NA>,Tharai Thappattai,0.0
4,Shôji Sakai,1001,movie,<NA>,<NA>,Tokyo Booty Nights,1.0
5,Shôji Sakai,1001,movie,<NA>,<NA>,Nausicaä of the Valley of the Wind,0.0
6,Shôji Sakai,1001,movie,<NA>,<NA>,The Strawberries and the Gun,1.0
7,Shôji Sakai,1001,movie,<NA>,<NA>,Sexy S.W.A.T. Team,0.0
8,Brahmanandam,807,movie,1956,<NA>,Ready,0.0
9,Brahmanandam,807,movie,1956,<NA>,Dookudu,0.0


## Analysis 2: Actors with More Runtime Done

In [62]:
# Actors with more runtime: Sum runtime from all participations
actors_runtime = (
    participations_pers.merge(
        dim_title_basic[['tconst', 'runtimeMinutes']], 
        on='tconst', 
        how='left'
    )
    .groupby('nconst')
    .agg({
        'runtimeMinutes': 'sum',
        'tconst': 'count'
    })
    .rename(columns={'runtimeMinutes': 'total_runtime_minutes', 'tconst': 'participation_count'})
    .reset_index()
    .sort_values('total_runtime_minutes', ascending=False, na_position='last')
    .merge(dim_person, on='nconst',  how='left')
    .dropna(subset=['total_runtime_minutes'])
    .head(20)
)
# 1. Merge all your data together
full_df = (
    actors_runtime[['primaryName', 'total_runtime_minutes', 'participation_count', 'birthYear', 'deathYear', 'nconst']]
    .merge(participations_pers[['nconst', 'tconst']], on='nconst', how='left')
    .merge(dim_title_basic[['tconst', 'primaryTitle', 'runtimeMinutes']], on='tconst', how='left')
)

# 2. Sort by nconst (the person) and runtimeMinutes (descending)
# 3. Group by nconst and take the top row for each
top_title_per_person = (
    full_df.sort_values(['nconst', 'runtimeMinutes'], ascending=[True, False])
    .groupby('nconst', as_index=False)
    .head(1)
    .reset_index(drop=True)
)
top_title_per_person = (
    top_title_per_person.sort_values('total_runtime_minutes', ascending=False)
    [['primaryName', 'primaryTitle', 'total_runtime_minutes', 'participation_count', 'birthYear', 'deathYear', 'primaryTitle', 'runtimeMinutes']]
    # .rename(columns={'primaryTitle': 'longestTitle', 'runtimeMinutes': 'longestTitleRuntimeMinutes'})
)
top_title_per_person = top_title_per_person.loc[:, ~top_title_per_person.columns.duplicated()]
top_title_per_person.to_parquet(f'{project_src}\\data\\gold\\long_runtime_title.parquet', index=False) 

In [63]:
top_title_per_person

,primaryName,primaryTitle,total_runtime_minutes,participation_count,birthYear,deathYear,runtimeMinutes
2,Ilaiyaraaja,Veera,65649.0,1006,1943,<NA>,240.0
9,Gérard Courant,Carnets Filmés (Liste Complète),61422.0,408,1951,<NA>,28643.0
16,Shôji Sakai,Dog Star,54455.0,1001,<NA>,<NA>,125.0
19,Innovativ Kultur,Logistics,51420.0,1,<NA>,<NA>,51420.0
18,Daniel Andersson,Logistics,51420.0,1,<NA>,<NA>,51420.0
17,Erika Magnusson,Logistics,51420.0,1,<NA>,<NA>,51420.0
0,William Shakespeare,Hamlet,46232.0,530,1564,1616,242.0
11,Dong-Chun Hyeon,Echo of Love and Death,45770.0,514,<NA>,<NA>,200.0
12,Laxmikant Shantaram Kudalkar,Narasimha,43917.0,467,1937,1998,214.0
6,Brahmanandam,New,42863.0,807,1956,<NA>,200.0


## Analysis 3: Participants with More Known Participations

In [24]:
# Participants with more known participations: using bridge_kwn_titles
# Count known titles per person (using bridge_kwn_titles if available)
known_participations = (
    bridge_kwn_titles.groupby('nconst')
    .size()
    .reset_index(name='known_title_count')
    .sort_values('known_title_count', ascending=False)
    .merge(dim_person, on='nconst', how='left')
    .head(20)
)

print("Top 20 Participants with Most Known Titles:")
print(known_participations)

Top 20 Participants with Most Known Titles:
       nconst  known_title_count            primaryName  birthYear  deathYear  \
0   nm0010174                  4        Casimiro Acosta       <NA>       <NA>   
1   nm0010158                  4          Alvaro Acosta       <NA>       <NA>   
2   nm0010157                  4           Alpha Acosta       1973       <NA>   
3   nm0010155                  4         Alberto Acosta       <NA>       <NA>   
4   nm0010154                  4          Albert Acosta       <NA>       <NA>   
5   nm0010153                  4         Adriana Acosta       <NA>       <NA>   
6   nm0010148                  4        Tony Acosta Jr.       <NA>       <NA>   
7   nm0010144                  4  Damián Acosta Esparza       <NA>       <NA>   
8   nm0000033                  4       Alfred Hitchcock       1899       1980   
9   nm0000032                  4        Charlton Heston       1923       2008   
10  nm0000031                  4      Katharine Hepburn       190

## Analysis 4: Participants with More Distinct Genres

In [50]:
# Participants with more distinct genres worked with
# Join participations with genres through title
participants_genres = (
    participations_pers.merge(bridge_genres, on='tconst', how='inner')
    .groupby('nconst')['genre_id']
    .nunique()
    .reset_index(name='distinct_genres_count')
    .sort_values('distinct_genres_count', ascending=False)
    .merge(dim_person[[ 'primaryName' , 'birthYear', 'deathYear', 'nconst']], on='nconst', how='left')
    .head(20)
)

# print("Top 20 Participants with Most Distinct Genres:")
# print(participants_genres)
participants_genres

,nconst,distinct_genres_count,primaryName,birthYear,deathYear
0,nm0001637,27,Vincent Price,1911,1993
1,nm0001682,25,Mickey Rooney,1920,2014
2,nm0000616,25,Eric Roberts,1956,<NA>
3,nm0000799,25,Edward Asner,1929,<NA>
4,nm0000191,24,Ewan McGregor,1971,<NA>
5,nm0000165,24,Ron Howard,1954,<NA>
6,nm0000277,24,Richard Attenborough,1923,2014
7,nm0000407,24,Vivica A. Fox,1964,<NA>
8,nm0000553,24,Liam Neeson,1952,<NA>
9,nm0000308,24,Ernest Borgnine,1917,2012
